In [ ]:
from sklearn.feature_selection import chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

import pandas as pd
import numpy as np


def train_and_evaluate(X,y,model,test_size=0.2,random_state=42,positive_label=1):
    """
    Train and evaluate a supplied classification model.
    """

    X_train_split, X_test_split, y_train_split, y_test_split = (
        train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y
        )
    )

    fitted_model = clone(model)

    fitted_model.fit(
        X_train_split,
        y_train_split
    )

    y_pred = fitted_model.predict(
        X_test_split
    )

    classes = list(fitted_model.classes_)

    if positive_label not in classes:
        raise ValueError(
            f"Positive label {positive_label!r} is not present "
            f"in the model classes: {classes}"
        )

    positive_class_index = classes.index(
        positive_label
    )

    y_pred_probabilities = fitted_model.predict_proba(
        X_test_split
    )[:, positive_class_index]

    y_test_binary = (
        np.asarray(y_test_split) == positive_label
    ).astype(int)

    return {
        "accuracy": accuracy_score(
            y_test_split,
            y_pred
        ),

        "recall": recall_score(
            y_test_split,
            y_pred,
            pos_label=positive_label,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y_test_binary,
            y_pred_probabilities
        )
    }

# Example of using FSA, here chi-sqaure is used
def chi_square_scorer(X, y):
    """
    Return Chi-square feature importance scores.
    """

    if (X < 0).any().any():
        raise ValueError(
            "Chi-square requires all feature values "
            "to be non-negative."
        )

    scores, _ = chi2(
        X,
        y
    )

    return scores


def iterative_backward_elimination(X,y,model,feature_scorer,threshold=0.75,test_size=0.2,random_state=42,positive_label=1):
    """
    Perform iterative backward feature elimination.

    The supplied feature_scorer must:
    1. Accept X and y.
    2. Return one numeric score per feature.
    3. Use larger values for more important features.

    At each iteration:
    - evaluate the current feature subset,
    - calculate feature scores,
    - store and print the scores,
    - remove the lowest-scoring feature,
    - stop if any performance metric falls below the threshold.
    """

    if not isinstance(X, pd.DataFrame):
        raise TypeError(
            "X must be a pandas DataFrame."
        )

    if X.empty:
        raise ValueError(
            "X must contain at least one feature."
        )

    if len(X) != len(y):
        raise ValueError(
            "X and y must contain the same number of rows."
        )

    if not callable(feature_scorer):
        raise TypeError(
            "feature_scorer must be callable."
        )

    if not 0 <= threshold <= 1:
        raise ValueError(
            "threshold must be between 0 and 1."
        )

    features = list(X.columns)

    performance_history = []

    while features:

        # ---------------------------------
        # Train and evaluate current subset
        # ---------------------------------
        metrics = train_and_evaluate(
            X=X[features],
            y=y,
            model=model,
            test_size=test_size,
            random_state=random_state,
            positive_label=positive_label
        )

        # ---------------------------------
        # Calculate current feature scores
        # ---------------------------------
        scores = feature_scorer(X[features],y)

        scores = np.asarray(
            scores,
            dtype=float
        ).reshape(-1)

        if len(scores) != len(features):
            raise ValueError(
                "feature_scorer must return exactly one "
                "score for each feature."
            )

        feature_scores = pd.Series(
            scores,
            index=features
        ).replace(
            [np.inf, -np.inf],
            np.nan
        ).fillna(0.0)

        # ---------------------------------
        # Store current iteration results
        # ---------------------------------
        performance_history.append(
            {
                "iteration": len(performance_history) + 1,

                "features": features.copy(),

                "feature_scores":
                    feature_scores.to_dict(),

                "number_of_features":
                    len(features),

                "accuracy":
                    metrics["accuracy"],

                "recall":
                    metrics["recall"],

                "roc_auc":
                    metrics["roc_auc"]
            }
        )

        # ---------------------------------
        # Print current iteration
        # ---------------------------------
        print("\n" + "-" * 60)

        print(
            f"Iteration: "
            f"{len(performance_history)}"
        )

        print(
            f"Number of features: "
            f"{len(features)}"
        )

        print("\nFeature Scores:")

        for feature, score in (
            feature_scores
            .sort_values(
                ascending=False
            )
            .items()
        ):
            print(
                f"{feature}: "
                f"{score:.6f}"
            )

        print(
            f"\nAccuracy: "
            f"{metrics['accuracy']:.4f}"
        )

        print(
            f"Recall: "
            f"{metrics['recall']:.4f}"
        )

        print(
            f"ROC-AUC: "
            f"{metrics['roc_auc']:.4f}"
        )

        # ---------------------------------
        # Check stopping threshold
        # ---------------------------------
        all_metrics_passed = (
            metrics["accuracy"] >= threshold
            and metrics["recall"] >= threshold
            and metrics["roc_auc"] >= threshold
        )

        if not all_metrics_passed:

            print(
                f"\nStopping: at least one metric "
                f"is below {threshold:.2f}."
            )

            break

        # ---------------------------------
        # Stop if one feature remains
        # ---------------------------------
        if len(features) == 1:

            print(
                "\nStopping: only one feature remains."
            )

            break

        # ---------------------------------
        # Remove least important feature
        # ---------------------------------
        least_important_feature = (
            feature_scores.idxmin()
        )

        print(
            f"\nRemoved feature: "
            f"{least_important_feature} "
            f"(score = "
            f"{feature_scores[least_important_feature]:.6f})"
        )

        features.remove(
            least_important_feature
        )

    return performance_history


def print_performance_history(model_name,performance_history):
    """
    Print the complete feature-elimination
    performance history for one model.
    """

    print("\n" + "=" * 60)

    print(
        f"Model: {model_name}"
    )

    print("=" * 60)

    for result in performance_history:

        print(
            f"\nIteration: "
            f"{result['iteration']}"
        )

        print(
            f"Number of features: "
            f"{result['number_of_features']}"
        )

        print(
            f"Features: "
            f"{result['features']}"
        )

        print("\nFeature Scores:")

        feature_scores = pd.Series(
            result["feature_scores"]
        ).sort_values(
            ascending=False
        )

        for feature, score in feature_scores.items():

            print(
                f"{feature}: "
                f"{score:.6f}"
            )

        print(
            f"\nAccuracy: "
            f"{result['accuracy']:.4f}"
        )

        print(
            f"Recall: "
            f"{result['recall']:.4f}"
        )

        print(
            f"ROC-AUC: "
            f"{result['roc_auc']:.4f}"
        )


def main(X, y):

    models = {

        "Random Forest":
            RandomForestClassifier(
                n_estimators=400,
                random_state=42
            ),

        "MLP":
            Pipeline(
                steps=[
                    (
                        "scaler",
                        StandardScaler()
                    ),
                    (
                        "classifier",
                        MLPClassifier(
                            hidden_layer_sizes=(64, 32),
                            activation="relu",
                            solver="adam",
                            max_iter=1000,
                            random_state=42
                        )
                    )
                ]
            ),

        "Naive Bayes":
            GaussianNB(),

        "Logistic Regression":
            Pipeline(
                steps=[
                    (
                        "scaler",
                        StandardScaler()
                    ),
                    (
                        "classifier",
                        LogisticRegression(
                            C=1.0,
                            max_iter=1000,
                            random_state=42
                        )
                    )
                ]
            ),

        "KNN":
            Pipeline(
                steps=[
                    (
                        "scaler",
                        StandardScaler()
                    ),
                    (
                        "classifier",
                        KNeighborsClassifier(
                            n_neighbors=21
                        )
                    )
                ]
            )
    }

    all_results = {}

    for model_name, model in models.items():

        print("\n" + "#" * 70)

        print(
            f"Running IBE with model: "
            f"{model_name}"
        )

        print("#" * 70)

        performance_history = (
            iterative_backward_elimination(
                X=X,
                y=y,
                model=model,
                feature_scorer=chi_square_scorer,
                threshold=0.75,
                test_size=0.2,
                random_state=42,
                positive_label=1
            )
        )

        all_results[
            model_name
        ] = performance_history

        print_performance_history(
            model_name,
            performance_history
        )

    return all_results

# Run the complete experiment
all_results = main(
    X_train,
    y_train
)